# Zero trust for agents, on the mesh you already run

*An agent is neither a user nor a service. It acts for a user, runs like a service, and its behaviour is not deterministic. The service mesh already authenticates it. This demo shows the part the mesh does not do: governing what the agent is allowed to **do** once authenticated. No second mesh, no second control plane: agentgateway runs as the **waypoint** inside Solo Istio ambient.*

| Objection | What the mesh already gives you | What the agentgateway waypoint adds |
|---|---|---|
| "An agent is a workload. The mesh does identity, mTLS and authz." | SPIFFE identity per workload, mTLS everywhere, L4 authorization | Authorization on **what the agent does**: per-tool, per-model, per-token |
| "It inherits a service account with far more access than any task needs." | Workload identity is coarse: one identity, all tasks | A **delegated token** that binds the human (`sub`) to the agent (`act`); policy needs both halves |
| "Its behaviour is not deterministic." | Nothing: the mesh sees connections, not intent | Tool-level allow lists, LLM budgets, GenAI telemetry on every call |
| "My board will ask for control mappings." | Mesh controls map to network segmentation | Every span carries SPIFFE caller, user, actor, tool, model, tokens, cost and the decision |


```mermaid
%%{init: {'theme':'dark', 'themeVariables': {'fontSize':'14px'}}}%%
flowchart LR
  U[Analyst / Admin<br/>Keycloak login] -->|OIDC at the edge| IG[agentgateway<br/>ingress]
  IG -->|user access token| A[castellan-assistant<br/>spiffe://.../sa/castellan-assistant]
  A -->|"user token + own SA token"| STS[agentgateway STS<br/>RFC 8693]
  STS -->|"delegated token<br/>sub=user, act=agent"| A
  A -->|HBONE mTLS| WP[agentgateway<br/>waypoint]
  WP -->|"per-tool authz<br/>SPIFFE + roles"| MCP[castellan-ops<br/>MCP server]
  WP -->|"budget, key injection"| LLM[OpenAI]
  WP -.traces.-> UI[Solo UI]
  Z[ztunnel] -.identity, mTLS.- A
  Z -.- WP
  Z -.- MCP
```

**The story in one line:** ztunnel proves *who* the agent is; the waypoint decides *what it may do*, for *which user*, and *how much*.


```mermaid
%%{init: {'theme':'dark', 'themeVariables': {'fontSize':'14px'}}}%%
sequenceDiagram
  autonumber
  actor U as Avery (analyst)
  participant IG as agentgateway ingress
  participant KC as Keycloak (IdP)
  participant A as castellan-assistant<br/>spiffe://.../sa/castellan-assistant
  participant STS as agentgateway STS
  participant WP as agentgateway waypoint<br/>(castellan-tools)
  participant C as castellan-clearance<br/>(specialist, one task)
  participant LLM as OpenAI
  participant MCP as castellan-ops MCP

  U->>IG: open chat
  IG-->>U: 302 to Keycloak (authorization code)
  U->>KC: sign in
  KC-->>IG: code -> access token (roles: mcp.reader)
  IG->>STS: rescope: exchange the IdP token (no actor)
  STS-->>IG: STS token  aud=castellan-assistant  roles, authorized_agents
  IG->>A: request + x-user-access-token (rescoped; the IdP token never reaches the agent)
  A->>STS: exchange: subject = rescoped token, actor = agent SA token
  STS-->>A: delegated JWT  sub=avery  act.sub=castellan-assistant  roles, authorized_agents
  A->>WP: chat completion (mTLS via ztunnel, no API key)
  WP->>WP: caller = SPIFFE castellan-assistant; budget check
  WP->>LLM: inject key, originate TLS
  LLM-->>WP: response + token usage
  WP-->>A: response (usage debited to the agent's budget)
  A->>WP: MCP tools/list (Bearer delegated JWT)
  WP->>WP: verify STS JWT; CEL: SPIFFE == castellan-assistant AND act in authorized_agents AND roles has mcp.reader
  WP->>MCP: tools/list
  MCP-->>WP: 4 tools
  WP-->>A: 2 tools (write and destructive filtered out)
  A->>WP: tools/call revoke_credentials
  WP-->>A: denied at the waypoint (never reaches the server)
  Note over A,C: clearance lookups are delegated
  A->>C: ask (Bearer A's delegated token), via the agents' waypoint
  C->>STS: exchange: subject = A's token, actor = C's SA token
  STS-->>C: delegated JWT  sub=avery  act.sub=clearance  act.act.sub=assistant
  C->>WP: tools/call get_clearance_status
  WP->>WP: chain must be assistant > clearance, both authorized by the human
  WP->>MCP: tools/call
```


## Prerequisites

A Kubernetes cluster (GKE or kind tested) in your current kubectl context, with the platform installed by `./scripts/install.sh`: the Solo distribution of Istio in ambient mode, Solo Enterprise for agentgateway with its token service, and the Solo UI. See the README. The notebook needs `OPENAI_API_KEY` in your environment; start Jupyter from the repo root so the cells run there.

Three browser tabs, opened by the port-forward cell in Part 0:

| Tab | URL | Users |
|---|---|---|
| Chat | http://localhost:8080 | `avery.analyst` (role `mcp.reader`), `morgan.admin` (role `mcp.admin`), password `Castellan-Demo-2026` |
| Shadow agent | http://localhost:8081 | same login, a different workload: the agent the users never authorized |
| (no tab) | castellan-clearance | a specialist agent the assistant calls for one task, clearance lookups; visible in the assistant's identity card |
| Solo UI | http://localhost:4000/age/ | built-in admin. Zoom the browser to 80% so the Tracing table fits. |
| Service graph | http://localhost:4000/ie/graph | same Solo UI, Istio view. Group by **Workload**, 15m window. |
| Keycloak | http://localhost:8180 | reached automatically by the login redirect |


In [ ]:
set +H; unset DEBUG
. scripts/00-env.sh
: "${OPENAI_API_KEY:?export OPENAI_API_KEY before starting Jupyter}"
echo "context: $(kubectl config current-context)   cluster name: $CLUSTER_NAME"
istioctl version --remote=false 2>/dev/null | head -1
kubectl get gatewayclass enterprise-agentgateway enterprise-agentgateway-waypoint
kubectl -n agentgateway-system get deploy enterprise-agentgateway solo-enterprise-ui -o custom-columns=NAME:.metadata.name,READY:.status.readyReplicas

# Before the meeting — Setup (about two minutes, run once)

*Four ambient namespaces. `castellan-idp` runs Keycloak (the enterprise IdP stand-in), `castellan-agents` runs three agents (the assistant, a clearance specialist it delegates one task to, and a shadow that nobody authorized) behind their own waypoint, `castellan-tools` runs the MCP server and owns the LLM egress, `castellan-ingress` terminates the human login. Nothing here is a policy about what the agent may do.*

`setup.yaml` deploys all of it, plus tracing to the Solo UI and the (still empty) budget enforcement on the LLM route.


In [ ]:
./scripts/build-setup.sh          # renders setup.yaml with your images and cluster name
kubectl apply -f setup.yaml
kubectl -n castellan-tools create secret generic openai-secret --from-literal=Authorization="Bearer $OPENAI_API_KEY" --dry-run=client -o yaml | kubectl apply -f -
for d in castellan-idp/keycloak castellan-tools/castellan-ops castellan-tools/agentgateway-waypoint castellan-agents/agentgateway-waypoint castellan-agents/castellan-assistant castellan-agents/castellan-clearance castellan-agents/shadow-assistant castellan-ingress/castellan-ingress; do
  kubectl -n ${d%/*} rollout status deploy/${d#*/} --timeout=180s | tail -1
done
# Keycloak was just (re)created with fresh signing keys; the STS and ext-auth cache IdP keys, so start them clean.
kubectl -n agentgateway-system rollout restart deploy/enterprise-agentgateway deploy/ext-auth-service-enterprise-agentgateway >/dev/null
kubectl -n agentgateway-system rollout status deploy/enterprise-agentgateway --timeout=180s | tail -1
kubectl -n agentgateway-system rollout status deploy/ext-auth-service-enterprise-agentgateway --timeout=180s | tail -1
# warm the LLM route once so the first live answer is fast
kubectl -n castellan-agents exec deploy/probe -- curl -s -m 60 -o /dev/null -w 'LLM warm-up -> HTTP %{http_code}\n' http://llm.castellan.internal/openai/v1/chat/completions -H 'Content-Type: application/json' -d '{"model":"gpt-4o-mini","messages":[{"role":"user","content":"hi"}],"max_tokens":3}'

# Part 0 — Open the tabs

*The waypoint is just another Gateway: same GatewayClass mechanism as the community waypoint, same HBONE listener, same istiod, same ztunnel. Only the L7 engine differs. The namespace label is the whole enrollment. No second control plane.*

In [ ]:
kubectl get gateway -A
kubectl get ns castellan-tools -o jsonpath='castellan-tools  use-waypoint: {.metadata.labels.istio\.io/use-waypoint}{"\n"}'

In [ ]:
./scripts/port-forwards.sh start      # resilient: each forward re-establishes itself if a connection drops
echo "chat      http://localhost:8080          ($DEMO_USER_READER / $DEMO_USER_ADMIN, password $DEMO_USER_PASSWORD)"
echo "shadow    http://localhost:8081          (same login, an agent the users never authorized)"
echo "solo ui   http://localhost:4000/age/"
echo "graph     http://localhost:4000/ie/graph" 

# Part 1 — What the mesh already solves, and where it stops

*Open the chat tab. You are redirected to Keycloak: the human identity is terminated at the agentgateway ingress, not inside the agent. Sign in as `avery.analyst`.*

**Read the "Who is acting" card** (click *show json* for the raw delegated token):

- **human token**: read the issuer and audience. It is not the corporate IdP token: the ingress exchanged it at the agentgateway STS for a token issued by the STS and scoped to `aud: castellan-assistant`, carrying only the claims we allow. If this agent leaks its copy, it is a short-lived token for one internal audience, not the user's corporate credential. The agent does not know any of this; you do.
- **agents the human authorized**: a claim from the IdP naming the agents this user consents to act for them. It rides along; it is enforced later.
- **this agent**: the workload's **SPIFFE identity**, issued by Istio through ztunnel. The agent never configured a certificate. That is the part of zero trust the mesh already covers.
- **acting on record**: the agent exchanged the rescoped token plus its *own* service account token at the STS (RFC 8693 delegation). The delegated token says *this human, through this agent, from this pod*.
- **delegates to / downstream token**: this agent hands one task, clearance lookups, to a specialist agent. The specialist exchanges again, and its token carries the chain `castellan-assistant → castellan-clearance` under the same human. Call graphs you did not plan for are still one auditable identity chain.

**Ask: "What is your identity?"** The answer comes from the token, and the LLM call behind it goes through the waypoint with **no API key** in the agent.

**Ask: "What tools do you have?"** All four, including `revoke_credentials`. The mesh authenticated the agent perfectly and still lets any human, through this agent, do anything the MCP server offers. That is the gap.


#### Same picture in the Solo UI
Solo UI tab, **Tracing**: open the latest `chat` span. `src.identity` is the SPIFFE caller, `gen_ai.request.model`, the token counts, and under `gen_ai` the exact `prompt` and `completion` text are on the span (the search box does not match prompt text; open the span). Open a `tools/list` span: `agent.actor` and `user.name` are the two halves of the delegated identity, recorded by the waypoint, not by the agent.

# Part 2 — Least privilege per tool: what the agent may *do*

*One MCP server, four tools across three risk tiers. Server-level access would grant all four. The waypoint filters `tools/list` and rejects `tools/call` per tool, and every rule needs **three** things: the workload (mTLS SPIFFE), the human's role (from the delegated token), and the human's consent to this agent (`authorized_agents`, also from the token).*

| Tool | Tier | analyst (`mcp.reader`) | admin (`mcp.admin`) | any other workload |
|---|---|---|---|---|
| `list_contracts` | read | allowed (assistant) | allowed (assistant) | denied |
| `get_clearance_status` | read | allowed, **only via the clearance specialist, only in the chain assistant → clearance** | same | denied |
| `approve_access_request` | write | denied | allowed | denied |
| `revoke_credentials` | destructive | denied | allowed | denied |


#### 1 · Per-tool authorization on the MCP backend
The MCP route already accepts only tokens minted by the STS and verified against its JWKS (part of `setup.yaml`), so everything under `jwt.` is trustworthy. Three rules, OR-ed. Each requires a specific workload identity, every actor in the chain to be one of the human's `authorized_agents`, **and** a role from the delegated token. The assistant gets `list_contracts` (readers and admins) and the write and destructive tools (admins). `get_clearance_status` belongs to the clearance specialist alone, and only when the token's chain reads `assistant → clearance`: the specialist cannot be used on its own.

In [ ]:
kubectl apply -f manifests/50-mcp-tool-policy.yaml
sleep 3; kubectl -n castellan-tools get enterpriseagentgatewaypolicy -o custom-columns='NAME:.metadata.name,ACCEPTED:.status.ancestors[0].conditions[0].status,ATTACHED:.status.ancestors[0].conditions[1].status'
sed -n '/matchExpressions/,$p' manifests/50-mcp-tool-policy.yaml

#### 2 · Back in the chat, still as Avery
**Ask: "What tools do you have?"** Two tools now.

**Click "Revoke e10412 (destructive)".** The assistant cannot: the tool is not in its list, so it never even attempts it.

**Click "Force revoke_credentials (bypass the model)".** This skips the model and sends `tools/call` straight to the MCP route with Avery's delegated token. The waypoint refuses it as an unknown tool, so nothing reaches the MCP server: the deny happens at the gateway, not in the model's manners.

**Ask: "What is the clearance status of employee e20877?"** Allowed, but the assistant did not call the tool. It delegated to `castellan-clearance` (see **delegates to** and the downstream token chain in the identity card), which exchanged the assistant's token for one that carries the chain and called `get_clearance_status`.


#### 3 · Same agent, different human
Click **sign out** in the header (it ends the Keycloak session too), sign in as `morgan.admin`.

**Ask: "What tools do you have?"** All four. **Click "Revoke e10412 (destructive)".** It executes. Same agent, same SPIFFE identity, same code. The only thing that changed is the human on the token.


#### 4 · Who may call whom
The agents' namespace has its own waypoint, so agent-to-agent calls get policy too. This rule says only the assistant's workload identity may call the clearance specialist; anyone else is refused on the mTLS peer certificate before a request is read.

In [ ]:
kubectl apply -f manifests/52-agent-chain-authz.yaml
sleep 3; kubectl -n castellan-agents get enterpriseagentgatewaypolicy clearance-callers -o jsonpath='{range .status.ancestors[*]}{range .conditions[*]}{.type}={.status} {end}{end}{"\n"}'
sed -n '/matchExpressions/,$p' manifests/52-agent-chain-authz.yaml

Ask **"Clearance of e20877"** again: still works, the assistant is the allowed caller. In the Solo UI, **Tracing**, the `tools/call get_clearance_status` span carries `agent.chain: castellan-assistant > castellan-clearance` together with `user.name`: two verified actors, one human, one line.


#### 5 · The human decides which agent may act for them
Open the **shadow agent** tab (http://localhost:8081). Same login, same human, a different workload from the same image. Its card is red: it obtained a delegated token that honestly names it as the actor, but that actor is not in the human's `authorized_agents`, so the waypoint returns **no tools**. The user's consent is a claim in the token and a condition in the rule. No agent code involved.


#### 6 · The evidence, in the Solo UI
**Tracing**, open Morgan's `tools/call` span: `src.identity` (the workload), `agent.actor` (the actor from the token), `user.name` (morgan.admin), `gen_ai.tool.name`, the route and the status, all on one span. That line is the control mapping: who, through what, did which action, and who allowed it.

Four gates were on display, all outside the agents: the workload identity from mTLS, the human's role, the human's consent to every agent in the chain, and who may call whom. Evaluated together on every tool, on every hop.


# Part 3 — Bounding a non-deterministic workload: LLM budgets per agent

*You cannot review every prompt. You can cap what an agent may spend. The budget subject here is the agent's **mesh identity**; the same CRD takes users, groups, models or dollars.*


#### Create the budget in the Solo UI
The LLM route already carries a budget-enforcement policy (scaffolding, `manifests/60-llm-budget-enforcement.yaml`); it does nothing until a budget exists. Budgets are created by the budget owner, not the platform team, so create it in the UI.

Solo UI tab, **Cost Management**, **Budgets** tab, **Create Budget**:

| Field | Value |
|---|---|
| Budget name | `castellan-assistant-budget` |
| Namespace | `castellan-tools` (the only budget-enforced namespace offered) |
| Entry name | `castellan-assistant-daily-tokens` |
| Subject | **Add subject pair**: dimension **User**, value `castellan-agents/castellan-assistant` (offered from observed traffic: the agent's mesh identity) |
| Amount / Unit | `2000` / **Tokens** |
| Rolling resets | **Daily (24 hours)** |
| When exceeded | **BLOCK** |

Click **Create Budget**. The list shows `2,000 tokens / DAILY · BLOCK`. Small on purpose (about two chat turns) so it trips live; real ones use monthly windows and USD with a model cost catalog.

What the UI just did is a Kubernetes object, which is why it survives, versions and audits like everything else:

In [ ]:
if kubectl -n castellan-tools get enterpriseagentgatewaybudget castellan-assistant-budget >/dev/null 2>&1; then
  kubectl -n castellan-tools get enterpriseagentgatewaybudget castellan-assistant-budget -o yaml | sed -n '/^spec:/,/^status:/p'
  kubectl -n castellan-tools get enterpriseagentgatewaybudget castellan-assistant-budget -o jsonpath='{range .status.conditions[*]}{.type}={.status} {end}{"\n"}'
else
  echo "no budget yet: create it in the Solo UI, or from the CLI:  kubectl apply -f manifests/61-llm-budget.yaml"
fi

#### Spend it, in the chat
Ask two or three questions (**"List active contracts"**, **"Clearance of e20877"**, anything). Usage is debited after each response; the turn that crosses the line completes, the next one comes back red:

> *The gateway blocked the LLM call: this agent's token budget is exhausted (HTTP 429).*

The agent has no budget logic. It rendered what the waypoint returned.

Solo UI, **Cost Management**: the **Budgets** tab lists `castellan-assistant-budget` (2,000 tokens / DAILY · BLOCK) and the dashboard attributes the spend to `castellan-agents/castellan-assistant`, the mesh identity. **Dashboard**: tokens by model for the whole run.


# Part 4 — Bounding what the agent may *say*: a data guard on the LLM path

*Prompts are data leaving the enclave. The waypoint inspects them before they reach an external model; the agent has no idea the guard exists.*

In [ ]:
kubectl apply -f manifests/55-llm-prompt-guard.yaml
sleep 3; kubectl -n castellan-tools get enterpriseagentgatewaypolicy llm-data-guard -o jsonpath='{range .status.ancestors[*]}{range .conditions[*]}{.type}={.status} {end}{end}{"\n"}'
sed -n '/promptGuard/,$p' manifests/55-llm-prompt-guard.yaml

In the chat (either user), click **"Send an SSN to the model"**. The turn comes back red: *The LLM call was refused: HTTP 403.* followed by the body *Blocked by Castellan data guard: a Social Security Number cannot be sent to an external model.* Ask anything else and it works. Solo UI, **Tracing**: the blocked call is a span with HTTP 403 and the guard's reason.


# Observe it all

*Every decision above is a span in the Solo UI and a log line on the waypoint: SPIFFE caller, user, actor, route, tool or model, tokens, cost, and the reason for a denial.*

Solo UI: **Gateways** lists the waypoint and the ingress under the same control plane as the Istio classes. **Policies** lists the policies you applied. **Tracing** filtered on the last 15 minutes is the audit trail of the demo.

#### Control mapping

| What you saw | Zero trust tenet (NIST SP 800-207) | CISA ZTMM pillar | Evidence on every span |
|---|---|---|---|
| Agent has a SPIFFE identity and mTLS with no code change | All communication is secured regardless of location (§2, tenet 2) | Networks; Applications & Workloads | `src.identity` |
| Human login terminated at the ingress, never inside the agent | Access granted per session, authenticated before it is granted (tenet 3, 6) | Identity | `user.name`, `jwt.sub` |
| Ingress rescopes the human token; agents never hold the corporate credential | Least-privilege credentials, minimize exposure (tenet 3) | Identity | `aud` on the token the agent received |
| Delegated token binds human to agent; unauthorized agent refused on the human's consent (`authorized_agents`) | Dynamic policy includes the requesting asset and the identity (tenet 4) | Identity; Applications & Workloads | `agent.actor`, tool rule |
| Per-tool allow list requiring workload identity **and** human role | Least-privilege, per-resource access (tenet 3) | Applications & Workloads; Data | `gen_ai.tool.name`, `http.status` |
| Agent-to-agent hop: chained delegation token, tool bound to the chain, caller allow list on the specialist | Per-session, per-resource trust; no implicit trust between workloads (tenet 3, 4) | Applications & Workloads | `agent.chain`, span on the agents' waypoint |
| Denied call never reaches the MCP server | Enforcement at the policy enforcement point, not the resource (§3) | Applications & Workloads | span `status`, empty server audit |
| Per-agent LLM budget | Continuous monitoring and measurement of asset behaviour (tenet 5) | Visibility & Analytics | `budget_id`, `gen_ai.usage.*` |
| Data guard on prompts | Protect data in transit to external resources (tenet 2) | Data | guard reason on the 403 span |
| Every decision traced with caller, user, actor, tool, model, cost | Collect and use information to improve posture (tenet 7) | Visibility & Analytics | the Tracing view |


#### The same evidence as log lines

In [ ]:
kubectl -n castellan-tools logs deploy/agentgateway-waypoint --tail=300 \
  | grep -E 'protocol=(mcp|llm)' | tail -8 \
  | grep -oE 'src.identity=[^ ]+|protocol=[^ ]+|mcp.method.name=[^ ]+|gen_ai.request.model=[^ ]+|http.status=[0-9]+|budget_action=[^ ]+|error=[^ ]+' \
  | tr '\n' ' ' | sed 's/ src.identity/\nsrc.identity/g'; echo

## Reset
Removes the budget (UI- or CLI-created), the demo policies and the budget counters, keeps the scaffolding so the notebook can be re-run from Part 1. Uncomment the last line to tear everything down.

In [ ]:
kubectl -n castellan-tools delete enterpriseagentgatewaybudget castellan-assistant-budget --ignore-not-found
kubectl delete -f manifests/50-mcp-tool-policy.yaml -f manifests/52-agent-chain-authz.yaml -f manifests/55-llm-prompt-guard.yaml --ignore-not-found
kubectl -n agentgateway-system exec deploy/ext-cache-enterprise-agentgateway-waypoint -- redis-cli FLUSHALL >/dev/null   # forget budget counters
./scripts/port-forwards.sh stop
# kubectl delete -f setup.yaml --ignore-not-found